# 图元计数流程演示

本笔记本演示计数流程的最小闭环：读取候选模式、加载目标图数据集、调用 `count_graphlets` 统计出现次数，并把结果保存为 JSON。

In [ ]:
import sys
from pathlib import Path

repo_root = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src").exists()), None)
if repo_root is None:
    raise RuntimeError("Could not locate repository root")
repo_root = str(repo_root)
sys.path.insert(0, repo_root)

import argparse
import json
import pickle

import networkx as nx
import torch_geometric.utils as pyg_utils

from src.core.cli import add_runtime_args, setup_runtime
from src.core import dataset_registry
from src.analyze.count_patterns import count_graphlets
from src.logger import RunLogger

In [ ]:
parser = argparse.ArgumentParser(description="计数流程演示")
parser.add_argument('--dataset', type=str)
parser.add_argument('--queries_path', type=str)
parser.add_argument('--out_path', type=str)
parser.add_argument('--count_method', type=str)
parser.add_argument('--max_queries', type=int)
parser.add_argument('--progress_every', type=int)
parser.add_argument('--node_anchored', action="store_true")
add_runtime_args(parser, include_gpu=False, include_seed=True, include_n_workers=True, include_progress_write_interval=True)
parser.set_defaults(dataset="enzymes", queries_path="results/enzymes_hq.p", out_path="results/notebook-counts.json", count_method="freq", max_queries=2, progress_every=1, n_workers=2, seed=0)
args = parser.parse_args("")
args.queries_path = str(Path(repo_root) / args.queries_path)
args.out_path = str(Path(repo_root) / args.out_path)
device = setup_runtime(args)
print({"device": str(device), "dataset": args.dataset, "queries_path": args.queries_path, "out_path": args.out_path})

In [ ]:
with open(args.queries_path, 'rb') as f:
    queries = pickle.load(f)
queries = queries[:args.max_queries]

normalized_dataset = dataset_registry.validate_dataset_name(args.dataset, "count")
args.dataset = normalized_dataset
dataset, _ = dataset_registry.load_dataset_for_stage(args.dataset, "count")
targets = []
for graph in list(dataset)[:20]:
    if not isinstance(graph, nx.Graph):
        graph = pyg_utils.to_networkx(graph).to_undirected()
    targets.append(graph)
print({"queries": len(queries), "targets": len(targets)})

In [ ]:
Path(args.out_path).parent.mkdir(parents=True, exist_ok=True)
query_lens = [len(q) for q in queries]
with RunLogger(args):
    n_matches = count_graphlets(queries, targets,
                                n_workers=args.n_workers,
                                method=args.count_method,
                                node_anchored=args.node_anchored,
                                progress_every=args.progress_every)
    with open(args.out_path, 'w') as f:
        json.dump((query_lens, n_matches, []), f)
print({"saved_to": args.out_path, "query_lens": query_lens, "counts": n_matches})